# Load a model

Pick one of the two ways below. Both end with the same `ata.run(...)` call.

| | Use it when |
|---|---|
| Local | You download the weights and run `llama-server` on this computer |
| Remote | Name and ML each have a running HTTP endpoint |

## Local

The weights are on the Hugging Face Hub: [lieberlee/annotune-qwen3.5-4b-gguf](https://huggingface.co/lieberlee/annotune-qwen3.5-4b-gguf). It holds the Q6_K base (converted from [Qwen/Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B), Apache-2.0) and the two LoRA adapters, in the folder layout `local_models` reads.

The first `run` starts `http://127.0.0.1:8080`. A later `run` reuses that server when the base and both adapters match.

### 1. Install the launcher

Do this once. Skip it when `llama-server` is already installed. The build is [llama.cpp](https://github.com/ggml-org/llama.cpp) **b12180**.

| System | Support |
|---|---|
| Linux with an NVIDIA GPU | Supported. Steps below. |
| macOS (Apple silicon or Intel) | Supported. Use the macOS steps below. The Metal build uses the GPU automatically. |
| Windows | Not supported for the local section. `import annotuneagent` works, but starting and stopping `llama-server` uses POSIX process calls. Run the server on Linux or macOS and use the Remote section, or use WSL. |

**Linux, CUDA 12.8.** Unpack the server and the CUDA runtime next to each other.

```bash
mkdir -p "$HOME/llama"
cd "$HOME/llama"
curl -L -o llama-cuda.tar.gz \
  https://github.com/ggml-org/llama.cpp/releases/download/b12180/llama-b12180-bin-ubuntu-cuda-12.8-x64.tar.gz
curl -L -o cudart.tar.gz \
  https://github.com/ggml-org/llama.cpp/releases/download/b12180/cudart-llama-b12180-bin-ubuntu-cuda-12.8-x64.tar.gz
tar -xzf llama-cuda.tar.gz
tar -xzf cudart.tar.gz
export LLAMA_SERVER="$HOME/llama/llama-b12180/llama-server"
"$LLAMA_SERVER" --version
```

The version line contains `build 12180`.

**macOS.** Pick `arm64` for Apple silicon and `x64` for Intel. There is no CUDA runtime to unpack.

```bash
mkdir -p "$HOME/llama"
cd "$HOME/llama"
curl -L -o llama-macos.tar.gz \
  https://github.com/ggml-org/llama.cpp/releases/download/b12180/llama-b12180-bin-macos-arm64.tar.gz
tar -xzf llama-macos.tar.gz
export LLAMA_SERVER="$HOME/llama/llama-b12180/llama-server"
xattr -dr com.apple.quarantine "$HOME/llama/llama-b12180"
"$LLAMA_SERVER" --version
```

The `xattr` line removes the download quarantine flag that otherwise stops macOS from running the binary. The base and both adapters need about 5 GB of memory with the default context.

### 2. Download the weights

About 3.7 GB. Only the three GGUF files and the license are fetched; the `_peft` folders (PEFT safetensors for further training) are skipped. Run it again after an interrupted download and it resumes.

`huggingface_hub` is installed with `pip install huggingface_hub`.

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

model_root = Path(snapshot_download(
    "lieberlee/annotune-qwen3.5-4b-gguf",
    local_dir=Path.home() / "annotune-models",
    allow_patterns=["base/*.gguf", "lora/*/*.gguf", "LICENSE", "NOTICE"],
))
print(model_root)

The files this leaves under `model_root`:

```text
model_root/base/Qwen3.5-4B-base.Q6_K.gguf
model_root/lora/name/annotune-v9h-name-checkpoint-412.lora.f16.gguf
model_root/lora/ml/annotune-v9e-ml-checkpoint-231.lora.f16.gguf
```

Check them against the published checksums:

In [ ]:
import hashlib

expected = {
    "base/Qwen3.5-4B-base.Q6_K.gguf": "eb569f4602efbb4d3aff68bb8ced3de86b9af638159227951b7b1ee84ffaf2c4",
    "lora/name/annotune-v9h-name-checkpoint-412.lora.f16.gguf": "eacdca50df3e94b17b46b3029807ba663383973de1e1391e5042a77fcacda51f",
    "lora/ml/annotune-v9e-ml-checkpoint-231.lora.f16.gguf": "65c3e122b485999390d44a6f87789c706e1d9152d8cc6088f5ae53c7c71c043f",
}
for name, digest in expected.items():
    h = hashlib.sha256()
    with open(model_root / name, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    print("ok " if h.hexdigest() == digest else "BAD", name)

### 3. Load

Set `LLAMA_SERVER` to the `llama-server` file from step 1, then run the cell. The tokenizer is installed with the package. `loaded:` is `http://127.0.0.1:8080` when a server is already running, and `None` when it is not. This cell does not start a server.

In [ ]:
import os

import annotuneagent as ata

os.environ["LLAMA_SERVER"] = "/path/to/llama-server"

print("loaded:", ata.local_server())
models = ata.local_models(model_root)
print("server:", models.name.server)

The server's state file and log live in a private folder: `$XDG_RUNTIME_DIR/annotune`, or `annotune-<user>` under the system temp folder. The folder is created with mode 0700. If it exists but belongs to another user, is a symlink, or is open to group or others, `run` stops with an error instead of reusing it. Delete or fix that folder and run again.

If a server on that address was started with different weights, `run` stops with `server already runs different GGUF, adapters, or context`. Stop that `llama-server` process and run again.

### 4. Annotate

`adata` comes from `ata.prepare`. Run this in the same notebook, after the cell above.

```python
report = ata.run(adata, "leiden", species="human", models=models)
print(ata.local_server())
```

The first call starts the server and leaves it running. The next call prints:

```text
server already loaded at http://127.0.0.1:8080; not loading again
```

`run` changes `adata` in place: it adds `annotune_*` columns and removes any stale `annotune_sub` column first. Models run on every cluster, not only the unresolved ones. The generated `annotation_code` in the report expects the annotated object (it asserts that the cluster column exists), not the raw input.

## Remote

Use this when name and ML each have a running endpoint. Set the addresses of the two model services. No local GGUF files are required.

In [ ]:
import annotuneagent as ata

name_url = "http://NAME_HOST:8080"
ml_url = "http://ML_HOST:8080"
models = ata.ModelChain(
    name=ata.TaskEndpoint(url=name_url),
    ml=ata.TaskEndpoint(url=ml_url),
)

Then annotate the same way as the local section:

```python
report = ata.run(adata, "leiden", species="human", models=models)
```

## Command line

Download the same files with the Hub CLI, then pass them to `annotuneagent`. `--server` is the address. An existing local server is reused only when its base, name and ML adapters match the paths below.

```bash
hf download lieberlee/annotune-qwen3.5-4b-gguf \
  --include "base/*.gguf" "lora/*/*.gguf" LICENSE NOTICE \
  --local-dir "$HOME/annotune-models"

export LLAMA_SERVER="$HOME/llama/llama-b12180/llama-server"
export ANNOTUNE_URL=http://127.0.0.1:8080
M="$HOME/annotune-models"
annotuneagent --input counts.h5ad --prepare --species human \
  --hypotheses hypotheses.json \
  --server "$ANNOTUNE_URL" \
  --base "$M/base/Qwen3.5-4B-base.Q6_K.gguf" \
  --name-adapter "$M/lora/name/annotune-v9h-name-checkpoint-412.lora.f16.gguf" \
  --ml-adapter "$M/lora/ml/annotune-v9e-ml-checkpoint-231.lora.f16.gguf" \
  --output result.json --output-h5ad labeled.h5ad
```

`--output-h5ad` is required when the adapters are set. The output folders are created and checked for write access before any model work starts, so a bad `--output` path fails at once. The report JSON is written before the h5ad.